# Interactive Batch GD Demo

Lecture-consistent linear-regression GD demo with matching notebook and HTML behavior.


### Optional dependency setup
Run this cell only if imports fail in your environment.

In [ ]:
import importlib.util

packages = [
    'numpy', 'matplotlib', 'plotly', 'ipywidgets', 'scipy', 'pandas', 'sklearn', 'seaborn'
]
missing = [p for p in packages if importlib.util.find_spec(p) is None]
if missing:
    print('Installing missing packages:', missing)
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(missing))
else:
    print('All common packages already available.')


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

METHOD = "batch"
METHOD_LABEL = "Batch GD"

def make_rng(seed=7):
    state = seed & 0xFFFFFFFF

    def rand():
        nonlocal state
        state = (1664525 * state + 1013904223) & 0xFFFFFFFF
        return state / 2**32

    return rand


def shuffled_indices(n, rand):
    idx = list(range(n))
    for i in range(n - 1, 0, -1):
        j = int(rand() * (i + 1))
        idx[i], idx[j] = idx[j], idx[i]
    return np.array(idx, dtype=int)


def solve_linear_fit(x, y):
    n = x.size
    sx = np.sum(x)
    sy = np.sum(y)
    sxx = np.sum(x * x)
    sxy = np.sum(x * y)
    det = n * sxx - sx * sx
    if abs(det) < 1e-12:
        return np.array([0.0, 0.0])
    return np.array([(sxx * sy - sx * sxy) / det, (n * sxy - sx * sy) / det], dtype=float)


def predict_line(theta, xv):
    return theta[0] + theta[1] * xv


def mse(theta, x, y):
    err = predict_line(theta, x) - y
    return float(np.mean(err * err))


def gradient(theta, idx, x, y):
    xb = x[idx]
    yb = y[idx]
    err = predict_line(theta, xb) - yb
    return np.array([2.0 * np.mean(err), 2.0 * np.mean(err * xb)], dtype=float)


N = 40
x = np.linspace(-2.3, 2.3, N)
noise_base = np.array([np.sin((i + 1) * 2.137) + 0.3 * np.cos((i + 1) * 0.917) for i in range(N)])

noise = widgets.FloatSlider(description='noise', min=0.05, max=1.0, step=0.05, value=0.25, readout_format='.2f', continuous_update=False)
alpha = widgets.FloatSlider(description='alpha', min=0.001, max=0.3, step=0.001, value=0.05, readout_format='.3f', continuous_update=False)
max_iter = widgets.IntSlider(description='max iter', min=1, max=200, step=1, value=10, continuous_update=False)
out = widgets.Output()


def optimize(method, x, y, alpha_value, steps, batch_size_value=8):
    theta = np.array([-1.8, 2.4], dtype=float)
    pathx = [float(theta[0])]
    pathy = [float(theta[1])]
    mses = []
    rand = make_rng(7)
    order = np.array([], dtype=int)
    ptr = 0

    def refill():
        nonlocal order, ptr
        order = shuffled_indices(len(x), rand)
        ptr = 0

    if method != 'batch':
        refill()

    for _ in range(steps):
        if method == 'batch':
            idx = np.arange(len(x), dtype=int)
        elif method == 'sgd':
            if ptr >= len(order):
                refill()
            idx = np.array([order[ptr]], dtype=int)
            ptr += 1
        else:
            if ptr >= len(order):
                refill()
            end = min(ptr + max(1, int(batch_size_value)), len(order))
            idx = order[ptr:end]
            ptr = end

        theta -= alpha_value * gradient(theta, idx, x, y)
        pathx.append(float(theta[0]))
        pathy.append(float(theta[1]))
        mses.append(mse(theta, x, y))

    return theta, pathx, pathy, mses


def render(*_):
    y = 1.2 + 0.8 * x - 0.6 * x * x + noise.value * noise_base
    theta_star = solve_linear_fit(x, y)
    batch_value = batch_size.value if METHOD == 'mini' else 8
    theta, pathx, pathy, mses = optimize(METHOD, x, y, alpha.value, max_iter.value, batch_value)

    xg = np.linspace(-2.3, 2.3, 300)
    y_star = predict_line(theta_star, xg)
    y_fit = predict_line(theta, xg)

    t0_min = min(min(pathx), float(theta_star[0])) - 1.0
    t0_max = max(max(pathx), float(theta_star[0])) + 1.0
    t1_min = min(min(pathy), float(theta_star[1])) - 1.0
    t1_max = max(max(pathy), float(theta_star[1])) + 1.0
    t0 = np.linspace(t0_min, t0_max, 70)
    t1 = np.linspace(t1_min, t1_max, 70)
    z = np.zeros((len(t1), len(t0)))
    for i, theta1 in enumerate(t1):
        for j, theta0 in enumerate(t0):
            z[i, j] = mse(np.array([theta0, theta1], dtype=float), x, y)

    fig = make_subplots(
        rows=1,
        cols=3,
        subplot_titles=(f'Batch GD: linear regression fit', 'MSE contour + trajectory', 'Optimization progress (full-batch MSE)')
    )
    fig.add_trace(go.Scatter(x=x, y=y, mode='markers', name='data', marker=dict(color='#1f77b4', size=7)), row=1, col=1)
    fig.add_trace(go.Scatter(x=xg, y=y_star, mode='lines', name='least-squares line', line=dict(color='#0f766e', dash='dash')), row=1, col=1)
    fig.add_trace(go.Scatter(x=xg, y=y_fit, mode='lines', name=f'Batch GD fit', line=dict(color='#dc2626', width=3)), row=1, col=1)

    fig.add_trace(go.Contour(x=t0, y=t1, z=z, colorscale='Viridis', showscale=False, name='MSE contour'), row=1, col=2)
    fig.add_trace(go.Scatter(x=pathx, y=pathy, mode='lines+markers', marker=dict(size=4), line=dict(color='#ef4444'), name='trajectory'), row=1, col=2)
    fig.add_trace(go.Scatter(x=[theta_star[0]], y=[theta_star[1]], mode='markers', marker=dict(color='#111827', size=10, symbol='x'), name='least-squares solution'), row=1, col=2)

    fig.add_trace(go.Scatter(x=np.arange(1, len(mses) + 1), y=mses, mode='lines', line=dict(color='#111'), name='MSE'), row=1, col=3)

    fig.update_xaxes(title_text='x', range=[-2.5, 2.5], row=1, col=1)
    fig.update_yaxes(title_text='y', range=[-4.5, 3.5], row=1, col=1)
    fig.update_xaxes(title_text='theta0', row=1, col=2)
    fig.update_yaxes(title_text='theta1', row=1, col=2)
    fig.update_xaxes(title_text='iteration', row=1, col=3)
    fig.update_yaxes(title_text='MSE', row=1, col=3)
    fig.update_layout(height=500, showlegend=True)

    with out:
        out.clear_output(wait=True)
        fig.show()
        status = f'method={METHOD} | alpha={alpha.value:.3f} | max_iter={max_iter.value}'
        if METHOD == 'mini':
            status = f'method={METHOD} | batch_size={batch_value} | alpha={alpha.value:.3f} | max_iter={max_iter.value}'
        print(status)
        print(f'final theta=({theta[0]:.3f}, {theta[1]:.3f})')
        print(f'least-squares theta=({theta_star[0]:.3f}, {theta_star[1]:.3f})')
        print(f'final full-batch MSE={mse(theta, x, y):.3f}')


for w in [noise, alpha, max_iter]:
    w.observe(render, 'value')
display(widgets.HBox([noise, alpha, max_iter]))
display(out)
render()
